# Phase 8: Exposure at Default (EAD) Modeling
This notebook implements Exposure at Default (EAD) modeling. EAD estimates the exposure amount (outstanding balance) at the time of default. Target is defined as: `EAD % = (funded_amnt - total_rec_prncp) / funded_amnt` capped to `[0.0, 1.0]`.


In [1]:
import pandas as pd
import numpy as np
import os
import sys

# Ensure we are running from the project root directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.append(os.path.abspath('src'))
from data_processing import DataProcessor
from ead_model import EADModel


## 1. Load Data Splits
Load the preprocessed default datasets.


In [2]:
processor = DataProcessor('data/loan.csv')
processor.clean_data()
train_df, oot_df = processor.split_data()
print('Train size:', train_df.shape)
print('OOT size:', oot_df.shape)


Train size: (18061, 116)
OOT size: (20516, 116)


## 2. Train EAD Model
Instantiate and fit the EAD XGBoost model on historical defaults.


In [3]:
ead_model = EADModel()
metrics = ead_model.fit(train_df, oot_df)
print('Model Evaluation Metrics:')
print(metrics)


Preparing EAD Training Data...
EAD Training cohort size: 2371 defaults
Preparing EAD Validation Data...
EAD Validation cohort size: 20516 defaults
Training EAD XGBoost Regressor...

EAD Model Evaluation Summary:
EAD XGBoost Validation - RMSE: 0.5674, MAE: 0.5428, R2: -3.8119
Model Evaluation Metrics:
{'train_rmse': np.float64(0.21214768164860814), 'train_mae': 0.17694997865125936, 'train_r2': 0.2911537552587091, 'val_rmse': np.float64(0.5674300090016404), 'val_mae': 0.5428316709507466, 'val_r2': -3.8118809869294115}


## 3. Generate Predictions & Reports
Calculate predictions for the OOT validation set, save the pickled model file, and compile the final PDF model report.


In [4]:
# Predict on OOT
oot_defaults = oot_df[oot_df['loan_status'].isin(['Charged Off', 'Default'])].copy()
oot_defaults['pred_ead_pct'] = ead_model.predict_ead(oot_defaults)
oot_defaults['actual_ead_pct'] = ead_model.calculate_ead_target(oot_defaults)
print(oot_defaults[['actual_ead_pct', 'pred_ead_pct']].describe())

# Save outputs
ead_model.save_model('outputs/scorecards/ead_model.pkl')
oot_defaults[['id', 'member_id', 'actual_ead_pct', 'pred_ead_pct']].to_csv('outputs/scorecards/ead_predictions.csv', index=False)
ead_model.generate_report(train_df, oot_df, 'outputs/reports/ead_model_report.pdf', metrics)
print('EAD predictions and PDF report generated successfully.')


       actual_ead_pct  pred_ead_pct
count     3256.000000   3256.000000
mean         0.657808      0.628498
std          0.239995      0.058363
min          0.012560      0.387997
25%          0.484649      0.590549
50%          0.707062      0.629108
75%          0.860205      0.665692
max          1.000000      0.884513
EAD Model successfully saved to outputs/scorecards/ead_model.pkl
EAD Model report successfully generated at outputs/reports/ead_model_report.pdf
EAD predictions and PDF report generated successfully.
